In [1]:
import argparse
import copy
import json
import os
import random
import time
import warnings
import wave
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


NUM_CLASSES = 10
BASELINE_DEFAULTS = {
    "dropout": 0.25,
    "mixup_alpha": 0.2,
    "mixup_prob": 0.25,
    "noise_prob": 0.35,
    "label_smoothing": 0.05,
    "patience": 4,
    "model_width": 32,
    "ema_decay": 0.0,
    "tta": 1,
    "speed_prob": 0.8,
    "speed_min_rate": 0.9,
    "speed_max_rate": 1.1,
    "shift_prob": 0.5,
    "shift_fraction": 0.1,
    "polarity_prob": 0.25,
    "drop_prob": 0.25,
    "drop_min_fraction": 0.02,
    "drop_max_fraction": 0.08,
    "gain_min": 0.75,
    "gain_max": 1.25,
    "snr_min_db": 10.0,
    "snr_max_db": 20.0,
    "tone_prob": 0.0,
    "clip_prob": 0.0,
    "spec_max_time_mask": 12,
    "spec_max_freq_mask": 8,
    "spec_time_masks": 2,
    "spec_freq_masks": 2,
}
ROBUST_V2_OVERRIDES = {
    "dropout": 0.3,
    "mixup_alpha": 0.25,
    "mixup_prob": 0.3,
    "noise_prob": 0.45,
    "label_smoothing": 0.06,
    "patience": 5,
    "model_width": 36,
    "ema_decay": 0.995,
    "tta": 3,
    "speed_prob": 0.85,
    "speed_min_rate": 0.9,
    "speed_max_rate": 1.1,
    "shift_prob": 0.6,
    "shift_fraction": 0.12,
    "polarity_prob": 0.25,
    "drop_prob": 0.28,
    "drop_min_fraction": 0.02,
    "drop_max_fraction": 0.1,
    "gain_min": 0.7,
    "gain_max": 1.3,
    "snr_min_db": 8.0,
    "snr_max_db": 18.0,
    "tone_prob": 0.2,
    "clip_prob": 0.1,
    "spec_max_time_mask": 14,
    "spec_max_freq_mask": 10,
    "spec_time_masks": 2,
    "spec_freq_masks": 2,
}


def default_device():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            return "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            return "cpu"


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--preset", choices=["baseline", "robust_v2"], default="baseline")
    parser.add_argument("--train-csv", default="train.csv")
    parser.add_argument("--test-csv", default="test.csv")
    parser.add_argument("--sample-submission", default="sample_submission.csv")
    parser.add_argument("--train-dir", default="train_audio")
    parser.add_argument("--test-dir", default="test_audio")
    parser.add_argument("--submission-path", default="submission.csv")
    parser.add_argument("--artifacts-dir", default="artifacts")
    parser.add_argument("--device", default=default_device())
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--num-folds", type=int, default=5)
    parser.add_argument("--folds-to-run", default="")
    parser.add_argument("--epochs", type=int, default=18)
    parser.add_argument("--patience", type=int, default=4)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--lr", type=float, default=3e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--grad-clip", type=float, default=5.0)
    parser.add_argument("--label-smoothing", type=float, default=0.05)
    parser.add_argument("--mixup-alpha", type=float, default=0.2)
    parser.add_argument("--mixup-prob", type=float, default=0.25)
    parser.add_argument("--noise-prob", type=float, default=0.35)
    parser.add_argument("--dropout", type=float, default=0.25)
    parser.add_argument("--model-width", type=int, default=32)
    parser.add_argument("--ema-decay", type=float, default=0.0)
    parser.add_argument("--tta", type=int, default=1)
    parser.add_argument("--target-sr", type=int, default=16000)
    parser.add_argument("--max-seconds", type=float, default=1.0)
    parser.add_argument("--n-mels", type=int, default=64)
    parser.add_argument("--n-fft", type=int, default=512)
    parser.add_argument("--win-length", type=int, default=400)
    parser.add_argument("--hop-length", type=int, default=160)
    parser.add_argument("--f-min", type=float, default=20.0)
    parser.add_argument("--f-max", type=float, default=7800.0)
    parser.add_argument("--speed-prob", type=float, default=0.8)
    parser.add_argument("--speed-min-rate", type=float, default=0.9)
    parser.add_argument("--speed-max-rate", type=float, default=1.1)
    parser.add_argument("--shift-prob", type=float, default=0.5)
    parser.add_argument("--shift-fraction", type=float, default=0.1)
    parser.add_argument("--polarity-prob", type=float, default=0.25)
    parser.add_argument("--drop-prob", type=float, default=0.25)
    parser.add_argument("--drop-min-fraction", type=float, default=0.02)
    parser.add_argument("--drop-max-fraction", type=float, default=0.08)
    parser.add_argument("--gain-min", type=float, default=0.75)
    parser.add_argument("--gain-max", type=float, default=1.25)
    parser.add_argument("--snr-min-db", type=float, default=10.0)
    parser.add_argument("--snr-max-db", type=float, default=20.0)
    parser.add_argument("--tone-prob", type=float, default=0.0)
    parser.add_argument("--clip-prob", type=float, default=0.0)
    parser.add_argument("--spec-max-time-mask", type=int, default=12)
    parser.add_argument("--spec-max-freq-mask", type=int, default=8)
    parser.add_argument("--spec-time-masks", type=int, default=2)
    parser.add_argument("--spec-freq-masks", type=int, default=2)
    parser.add_argument("--max-train-samples", type=int, default=0)
    parser.add_argument("--max-test-samples", type=int, default=0)
    return parser.parse_args()


def apply_preset(args):
    if args.preset == "baseline":
        return args
    if args.preset != "robust_v2":
        raise ValueError(f"Unsupported preset: {args.preset}")
    for key, value in ROBUST_V2_OVERRIDES.items():
        if getattr(args, key) == BASELINE_DEFAULTS[key]:
            setattr(args, key, value)
    return args


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def resolve_audio_dir(path_str):
    root = Path(path_str)
    candidates = [root]
    if root.exists():
        candidates.extend(child for child in root.iterdir() if child.is_dir())
        candidates.append(root / root.name)
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*.wav")):
            return candidate
    raise FileNotFoundError(f"Could not find a directory with .wav files under {root}")


def load_ids_from_sources(test_csv, sample_submission, test_dir):
    for csv_path in [test_csv, sample_submission]:
        path = Path(csv_path)
        if path.exists():
            frame = pd.read_csv(path)
            if "id" in frame.columns and len(frame) > 0:
                return frame["id"].astype(str).tolist()
    return [path.stem for path in sorted(Path(test_dir).glob("*.wav"))]


def read_wav(path):
    with wave.open(str(path), "rb") as handle:
        sample_rate = handle.getframerate()
        channels = handle.getnchannels()
        sample_width = handle.getsampwidth()
        frame_count = handle.getnframes()
        raw_audio = handle.readframes(frame_count)
    if sample_width != 2:
        raise ValueError(f"Only 16-bit PCM is supported, got sample width {sample_width} in {path}")
    audio = np.frombuffer(raw_audio, dtype="<i2").astype(np.float32)
    if channels > 1:
        audio = audio.reshape(-1, channels).mean(axis=1)
    audio /= 32768.0
    return torch.from_numpy(audio), sample_rate


def resample_waveform(waveform, original_sr, target_sr):
    if original_sr == target_sr:
        return waveform
    new_length = max(1, int(round(waveform.numel() * target_sr / original_sr)))
    return F.interpolate(
        waveform.view(1, 1, -1),
        size=new_length,
        mode="linear",
        align_corners=False,
    ).view(-1)


def crop_or_pad(waveform, target_length, training):
    length = waveform.numel()
    if length > target_length:
        max_start = length - target_length
        start = random.randint(0, max_start) if training else max_start // 2
        return waveform[start : start + target_length]
    if length < target_length:
        padding = target_length - length
        left = random.randint(0, padding) if training else padding // 2
        right = padding - left
        return F.pad(waveform, (left, right))
    return waveform


def rms(waveform):
    return waveform.pow(2).mean().sqrt()


def speed_perturb(waveform, min_rate=0.9, max_rate=1.1):
    rate = random.uniform(min_rate, max_rate)
    return resample_by_rate(waveform, rate)


def resample_by_rate(waveform, rate):
    new_length = max(16, int(round(waveform.numel() / rate)))
    return F.interpolate(
        waveform.view(1, 1, -1),
        size=new_length,
        mode="linear",
        align_corners=False,
    ).view(-1)


def shift_with_padding(waveform, shift):
    if shift == 0:
        return waveform
    padded = torch.zeros_like(waveform)
    if shift > 0:
        padded[shift:] = waveform[:-shift]
    else:
        padded[:shift] = waveform[-shift:]
    return padded


def moving_average_filter(waveform, kernel_size):
    if kernel_size <= 1:
        return waveform
    kernel = torch.ones(1, 1, kernel_size, dtype=waveform.dtype, device=waveform.device) / kernel_size
    padded = F.pad(waveform.view(1, 1, -1), (kernel_size // 2, kernel_size // 2), mode="reflect")
    return F.conv1d(padded, kernel).view(-1)


def apply_tone_augmentation(waveform):
    kernel_size = random.choice([3, 5, 7, 9, 11, 15])
    smoothed = moving_average_filter(waveform, kernel_size)
    if random.random() < 0.5:
        mix = random.uniform(0.35, 0.75)
        return waveform.lerp(smoothed, mix)
    emphasis = random.uniform(0.2, 0.7)
    return waveform + emphasis * (waveform - smoothed)


def apply_waveform_augmentation(waveform, target_length, args, noise_waveform=None):
    if random.random() < args.speed_prob:
        waveform = speed_perturb(waveform, args.speed_min_rate, args.speed_max_rate)
    waveform = crop_or_pad(waveform, target_length, training=True)

    if noise_waveform is not None:
        if random.random() < 0.5:
            noise_waveform = speed_perturb(noise_waveform, min_rate=0.95, max_rate=1.05)
        noise_waveform = crop_or_pad(noise_waveform, target_length, training=True)
        clean_rms = rms(waveform).clamp_min(1e-4)
        noise_rms = rms(noise_waveform).clamp_min(1e-4)
        snr_db = random.uniform(args.snr_min_db, args.snr_max_db)
        scale = clean_rms / (noise_rms * (10 ** (snr_db / 20.0)))
        waveform = waveform + noise_waveform * scale

    if random.random() < args.tone_prob:
        waveform = apply_tone_augmentation(waveform)

    if random.random() < args.shift_prob:
        shift_limit = max(1, int(args.shift_fraction * target_length))
        waveform = shift_with_padding(waveform, shift=random.randint(-shift_limit, shift_limit))

    if random.random() < args.polarity_prob:
        waveform = -waveform

    if random.random() < args.drop_prob:
        min_drop = max(1, int(args.drop_min_fraction * target_length))
        max_drop = max(min_drop, int(args.drop_max_fraction * target_length))
        drop_length = random.randint(min_drop, max_drop)
        drop_start = random.randint(0, target_length - drop_length)
        waveform = waveform.clone()
        waveform[drop_start : drop_start + drop_length] = 0.0

    waveform = waveform * random.uniform(args.gain_min, args.gain_max)
    if random.random() < args.clip_prob:
        clip_level = random.uniform(0.3, 0.85)
        waveform = waveform.clamp(-clip_level, clip_level) / clip_level
    return waveform.clamp_(-1.0, 1.0)


class SpokenDigitDataset(Dataset):
    def __init__(self, ids, labels, audio_dir, target_sr, max_seconds, train, noise_prob, args):
        self.ids = list(ids)
        self.labels = None if labels is None else [int(label) for label in labels]
        self.audio_dir = Path(audio_dir)
        self.target_sr = target_sr
        self.target_length = int(round(target_sr * max_seconds))
        self.train = train
        self.noise_prob = noise_prob
        self.args = args

    def __len__(self):
        return len(self.ids)

    def _audio_path(self, sample_id):
        sample_id = str(sample_id)
        filename = sample_id if sample_id.endswith(".wav") else f"{sample_id}.wav"
        return self.audio_dir / filename

    def _load_waveform(self, index):
        waveform, sample_rate = read_wav(self._audio_path(self.ids[index]))
        return resample_waveform(waveform, sample_rate, self.target_sr)

    def __getitem__(self, index):
        waveform = self._load_waveform(index)
        if self.train:
            noise_waveform = None
            if self.noise_prob > 0 and len(self.ids) > 1 and random.random() < self.noise_prob:
                noise_index = random.randrange(len(self.ids) - 1)
                if noise_index >= index:
                    noise_index += 1
                noise_waveform = self._load_waveform(noise_index)
            waveform = apply_waveform_augmentation(waveform, self.target_length, self.args, noise_waveform)
        else:
            waveform = crop_or_pad(waveform, self.target_length, training=False)

        return {
            "id": self.ids[index],
            "waveform": waveform,
            "label": -1 if self.labels is None else self.labels[index],
        }


def collate_batch(batch):
    return {
        "ids": [item["id"] for item in batch],
        "waveforms": torch.stack([item["waveform"] for item in batch]),
        "labels": torch.tensor([item["label"] for item in batch], dtype=torch.long),
    }


def hz_to_mel(frequency):
    return 2595.0 * torch.log10(torch.tensor(1.0) + frequency / 700.0)


def mel_to_hz(mel_value):
    return 700.0 * (10 ** (mel_value / 2595.0) - 1.0)


def build_mel_filter(sample_rate, n_fft, n_mels, f_min, f_max):
    f_max = min(f_max, sample_rate / 2)
    fft_frequencies = torch.linspace(0, sample_rate / 2, n_fft // 2 + 1)
    mel_points = torch.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels + 2)
    hz_points = mel_to_hz(mel_points)
    filterbank = torch.zeros(n_mels, n_fft // 2 + 1)

    for mel_bin in range(n_mels):
        left = hz_points[mel_bin]
        center = hz_points[mel_bin + 1]
        right = hz_points[mel_bin + 2]
        up_slope = (fft_frequencies - left) / (center - left + 1e-8)
        down_slope = (right - fft_frequencies) / (right - center + 1e-8)
        filterbank[mel_bin] = torch.clamp(torch.minimum(up_slope, down_slope), min=0.0)

    return filterbank / filterbank.sum(dim=1, keepdim=True).clamp_min(1e-8)


class LogMelFrontend(nn.Module):
    def __init__(self, sample_rate, n_fft, win_length, hop_length, n_mels, f_min, f_max):
        super().__init__()
        self.n_fft = n_fft
        self.win_length = win_length
        self.hop_length = hop_length
        self.register_buffer("window", torch.hann_window(win_length), persistent=False)
        self.register_buffer(
            "mel_filter",
            build_mel_filter(sample_rate, n_fft, n_mels, f_min, f_max),
            persistent=False,
        )

    def forward(self, waveforms):
        spectrum = torch.stft(
            waveforms,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=self.window,
            center=True,
            return_complex=True,
        )
        power = spectrum.abs().pow(2)
        mel = torch.einsum("mf,bft->bmt", self.mel_filter, power)
        log_mel = torch.log(mel.clamp_min(1e-5))
        mean = log_mel.mean(dim=(-2, -1), keepdim=True)
        std = log_mel.std(dim=(-2, -1), keepdim=True).clamp_min(1e-4)
        log_mel = (log_mel - mean) / std

        delta = F.pad(log_mel[:, :, 1:] - log_mel[:, :, :-1], (1, 0))
        delta2 = F.pad(delta[:, :, 1:] - delta[:, :, :-1], (1, 0))
        return torch.stack([log_mel, delta, delta2], dim=1)


class SpecAugment(nn.Module):
    def __init__(self, max_time_mask=12, max_freq_mask=8, time_masks=2, freq_masks=2):
        super().__init__()
        self.max_time_mask = max_time_mask
        self.max_freq_mask = max_freq_mask
        self.time_masks = time_masks
        self.freq_masks = freq_masks

    def forward(self, features):
        if not self.training:
            return features
        features = features.clone()
        batch_size, _, freq_bins, time_steps = features.shape
        for index in range(batch_size):
            for _ in range(self.freq_masks):
                width = random.randint(0, min(self.max_freq_mask, freq_bins))
                if width == 0 or width >= freq_bins:
                    continue
                start = random.randint(0, freq_bins - width)
                features[index, :, start : start + width, :] = 0.0
            for _ in range(self.time_masks):
                width = random.randint(0, min(self.max_time_mask, time_steps))
                if width == 0 or width >= time_steps:
                    continue
                start = random.randint(0, time_steps - width)
                features[index, :, :, start : start + width] = 0.0
        return features


class SqueezeExcite(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(8, channels // reduction)
        self.layers = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, kernel_size=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, inputs):
        return inputs * self.layers(inputs)


class ConvNormAct(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super().__init__(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=kernel_size // 2,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, drop_prob=0.0):
        super().__init__()
        self.conv1 = ConvNormAct(in_channels, out_channels, stride=stride)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.se = SqueezeExcite(out_channels)
        self.dropout = nn.Dropout2d(drop_prob) if drop_prob > 0 else nn.Identity()
        self.shortcut = (
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
            if stride != 1 or in_channels != out_channels
            else nn.Identity()
        )
        self.activation = nn.SiLU(inplace=True)

    def forward(self, inputs):
        residual = self.shortcut(inputs)
        outputs = self.conv1(inputs)
        outputs = self.conv2(outputs)
        outputs = self.se(outputs)
        outputs = self.dropout(outputs)
        outputs = outputs + residual
        return self.activation(outputs)


class SpectrogramCNN(nn.Module):
    def __init__(self, num_classes, dropout, width):
        super().__init__()
        self.stem = ConvNormAct(3, width)
        self.stage1 = nn.Sequential(
            ResidualBlock(width, width),
            ResidualBlock(width, width),
        )
        self.stage2 = nn.Sequential(
            ResidualBlock(width, width * 2, stride=2, drop_prob=0.05),
            ResidualBlock(width * 2, width * 2),
        )
        self.stage3 = nn.Sequential(
            ResidualBlock(width * 2, width * 3, stride=2, drop_prob=0.08),
            ResidualBlock(width * 3, width * 3),
        )
        self.stage4 = nn.Sequential(
            ResidualBlock(width * 3, width * 4, stride=2, drop_prob=0.10),
            ResidualBlock(width * 4, width * 4),
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(width * 8, num_classes)

    def forward(self, inputs):
        outputs = self.stem(inputs)
        outputs = self.stage1(outputs)
        outputs = self.stage2(outputs)
        outputs = self.stage3(outputs)
        outputs = self.stage4(outputs)
        avg_pool = F.adaptive_avg_pool2d(outputs, 1).flatten(1)
        max_pool = F.adaptive_max_pool2d(outputs, 1).flatten(1)
        outputs = self.dropout(torch.cat([avg_pool, max_pool], dim=1))
        return self.classifier(outputs)


class SpokenDigitModel(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.frontend = LogMelFrontend(
            sample_rate=args.target_sr,
            n_fft=args.n_fft,
            win_length=args.win_length,
            hop_length=args.hop_length,
            n_mels=args.n_mels,
            f_min=args.f_min,
            f_max=args.f_max,
        )
        self.spec_augment = SpecAugment(
            max_time_mask=args.spec_max_time_mask,
            max_freq_mask=args.spec_max_freq_mask,
            time_masks=args.spec_time_masks,
            freq_masks=args.spec_freq_masks,
        )
        self.backbone = SpectrogramCNN(
            num_classes=NUM_CLASSES,
            dropout=args.dropout,
            width=args.model_width,
        )

    def forward(self, waveforms):
        features = self.frontend(waveforms)
        features = self.spec_augment(features)
        return self.backbone(features)


def make_stratified_folds(labels, num_folds, seed):
    if num_folds < 2:
        raise ValueError("num_folds must be at least 2")
    folds = [[] for _ in range(num_folds)]
    by_class = {}
    for index, label in enumerate(labels):
        by_class.setdefault(int(label), []).append(index)
    rng = random.Random(seed)
    for label in sorted(by_class):
        indices = by_class[label]
        rng.shuffle(indices)
        for offset, index in enumerate(indices):
            folds[offset % num_folds].append(index)
    return [sorted(fold) for fold in folds]


def smooth_targets(labels, num_classes, smoothing):
    targets = torch.full(
        (labels.size(0), num_classes),
        fill_value=smoothing / max(1, num_classes - 1),
        device=labels.device,
    )
    targets.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
    return targets


def soft_cross_entropy(logits, targets):
    return -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()


def maybe_mixup(waveforms, targets, alpha, probability):
    if alpha <= 0 or waveforms.size(0) < 2 or random.random() > probability:
        return waveforms, targets
    lam = np.random.beta(alpha, alpha)
    permutation = torch.randperm(waveforms.size(0), device=waveforms.device)
    mixed_waveforms = lam * waveforms + (1.0 - lam) * waveforms[permutation]
    mixed_targets = lam * targets + (1.0 - lam) * targets[permutation]
    return mixed_waveforms, mixed_targets


def create_loader(dataset, batch_size, shuffle, workers, seed, pin_memory):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=pin_memory,
        persistent_workers=workers > 0,
        collate_fn=collate_batch,
        worker_init_fn=seed_worker,
        generator=generator,
    )


def create_grad_scaler(device):
    enabled = device.type == "cuda"
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def create_autocast_context(device):
    if device.type != "cuda":
        return nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda")
    return torch.cuda.amp.autocast()


class ModelEma(nn.Module):
    def __init__(self, model, decay):
        super().__init__()
        self.decay = decay
        self.num_updates = 0
        self.module = copy.deepcopy(model).eval()
        for parameter in self.module.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.num_updates += 1
        decay = min(self.decay, (1 + self.num_updates) / (10 + self.num_updates))
        ema_state = self.module.state_dict()
        model_state = model.state_dict()
        for key, value in ema_state.items():
            source = model_state[key].detach()
            if not torch.is_floating_point(value):
                value.copy_(source)
                continue
            value.mul_(decay).add_(source, alpha=1.0 - decay)


def train_one_epoch(model, ema_model, loader, optimizer, scheduler, scaler, device, args):
    model.train()
    use_amp = device.type == "cuda"
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in loader:
        waveforms = batch["waveforms"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        targets = smooth_targets(labels, NUM_CLASSES, args.label_smoothing)
        waveforms, targets = maybe_mixup(waveforms, targets, args.mixup_alpha, args.mixup_prob)

        optimizer.zero_grad(set_to_none=True)
        autocast_context = create_autocast_context(device)
        with autocast_context:
            logits = model(waveforms)
            loss = soft_cross_entropy(logits, targets)

        if use_amp:
            scale_before = scaler.get_scale()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer_ran = scaler.get_scale() >= scale_before
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()
            optimizer_ran = True

        if optimizer_ran:
            scheduler.step()
            if ema_model is not None:
                ema_model.update(model)

        total_loss += loss.item() * waveforms.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += waveforms.size(0)

    return {
        "loss": total_loss / max(1, total_samples),
        "accuracy": total_correct / max(1, total_samples),
    }


@torch.no_grad()
def build_tta_waveform(waveforms, variant_index):
    if variant_index == 0:
        return waveforms
    target_length = waveforms.size(-1)
    if variant_index == 1:
        shift = max(1, int(0.04 * target_length))
        return torch.stack([shift_with_padding(waveform, shift) for waveform in waveforms], dim=0)
    if variant_index == 2:
        shift = max(1, int(0.04 * target_length))
        return torch.stack([shift_with_padding(waveform, -shift) for waveform in waveforms], dim=0)
    if variant_index == 3:
        return torch.stack(
            [crop_or_pad(resample_by_rate(waveform, 0.97), target_length, training=False) for waveform in waveforms],
            dim=0,
        )
    if variant_index == 4:
        return torch.stack(
            [crop_or_pad(resample_by_rate(waveform, 1.03), target_length, training=False) for waveform in waveforms],
            dim=0,
        )
    raise ValueError(f"Unsupported TTA variant: {variant_index}")


@torch.no_grad()
def evaluate(model, loader, device, tta=1):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    probabilities = []
    ids = []
    labels_out = []

    for batch in loader:
        waveforms = batch["waveforms"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        logits_sum = None
        for variant_index in range(max(1, tta)):
            augmented_waveforms = build_tta_waveform(waveforms, variant_index) if variant_index > 0 else waveforms
            logits = model(augmented_waveforms)
            logits_sum = logits if logits_sum is None else logits_sum + logits
        logits = logits_sum / max(1, tta)
        probs = logits.softmax(dim=1)
        probabilities.append(probs.cpu().numpy())
        ids.extend(batch["ids"])

        if torch.all(labels >= 0):
            loss = F.cross_entropy(logits, labels)
            total_loss += loss.item() * waveforms.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += waveforms.size(0)
            labels_out.append(labels.cpu().numpy())

    metrics = {
        "loss": total_loss / max(1, total_samples),
        "accuracy": total_correct / max(1, total_samples),
    }
    labels_array = np.concatenate(labels_out) if labels_out else None
    probs_array = np.concatenate(probabilities) if probabilities else np.empty((0, NUM_CLASSES), dtype=np.float32)
    return metrics, ids, probs_array, labels_array


def parse_folds_to_run(folds_to_run, num_folds):
    if not folds_to_run.strip():
        return list(range(num_folds))
    selected = sorted({int(item.strip()) for item in folds_to_run.split(",") if item.strip()})
    invalid = [item for item in selected if item < 0 or item >= num_folds]
    if invalid:
        raise ValueError(f"Invalid fold indices: {invalid}")
    return selected


def save_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)


def main():
    args = apply_preset(parse_args())
    if args.tta < 1 or args.tta > 5:
        raise ValueError("--tta must be between 1 and 5")
    set_seed(args.seed)

    train_dir = resolve_audio_dir(args.train_dir)
    test_dir = resolve_audio_dir(args.test_dir)
    artifacts_dir = Path(args.artifacts_dir)
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    train_df = pd.read_csv(args.train_csv)
    if args.max_train_samples and args.max_train_samples < len(train_df):
        train_df = train_df.sample(n=args.max_train_samples, random_state=args.seed).sort_values("id").reset_index(drop=True)

    test_ids = load_ids_from_sources(args.test_csv, args.sample_submission, test_dir)
    if args.max_test_samples and args.max_test_samples < len(test_ids):
        test_ids = test_ids[: args.max_test_samples]
    if not test_ids:
        raise ValueError("No test ids were found from CSV files or the test audio directory")

    labels = train_df["label"].astype(int).tolist()
    folds = make_stratified_folds(labels, args.num_folds, args.seed)
    folds_to_run = parse_folds_to_run(args.folds_to_run, args.num_folds)
    device = torch.device(args.device)
    pin_memory = device.type == "cuda"

    test_dataset = SpokenDigitDataset(
        ids=test_ids,
        labels=None,
        audio_dir=test_dir,
        target_sr=args.target_sr,
        max_seconds=args.max_seconds,
        train=False,
        noise_prob=0.0,
        args=args,
    )
    test_loader = create_loader(
        test_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        workers=args.workers,
        seed=args.seed,
        pin_memory=pin_memory,
    )

    oof_probabilities = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)
    oof_mask = np.zeros(len(train_df), dtype=bool)
    test_probabilities = np.zeros((len(test_ids), NUM_CLASSES), dtype=np.float32)
    fold_summaries = []

    print(f"Training on {len(train_df)} samples across folds {folds_to_run} using device={device}")
    print(f"Resolved audio directories: train={train_dir} test={test_dir}")

    for fold_index, validation_indices in enumerate(folds):
        if fold_index not in folds_to_run:
            continue

        validation_mask = np.zeros(len(train_df), dtype=bool)
        validation_mask[validation_indices] = True
        training_indices = np.flatnonzero(~validation_mask)

        train_split = train_df.iloc[training_indices].reset_index(drop=True)
        validation_split = train_df.iloc[validation_indices].reset_index(drop=True)

        train_dataset = SpokenDigitDataset(
            ids=train_split["id"].tolist(),
            labels=train_split["label"].tolist(),
            audio_dir=train_dir,
            target_sr=args.target_sr,
            max_seconds=args.max_seconds,
            train=True,
            noise_prob=args.noise_prob,
            args=args,
        )
        validation_dataset = SpokenDigitDataset(
            ids=validation_split["id"].tolist(),
            labels=validation_split["label"].tolist(),
            audio_dir=train_dir,
            target_sr=args.target_sr,
            max_seconds=args.max_seconds,
            train=False,
            noise_prob=0.0,
            args=args,
        )

        train_loader = create_loader(
            train_dataset,
            batch_size=args.batch_size,
            shuffle=True,
            workers=args.workers,
            seed=args.seed + fold_index,
            pin_memory=pin_memory,
        )
        validation_loader = create_loader(
            validation_dataset,
            batch_size=args.batch_size,
            shuffle=False,
            workers=args.workers,
            seed=args.seed + fold_index,
            pin_memory=pin_memory,
        )

        model = SpokenDigitModel(args).to(device)
        ema_model = ModelEma(model, decay=args.ema_decay) if args.ema_decay > 0 else None
        optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=args.lr,
            epochs=args.epochs,
            steps_per_epoch=max(1, len(train_loader)),
            pct_start=0.1,
        )
        scaler = create_grad_scaler(device)

        best_accuracy = -1.0
        best_epoch = 0
        epochs_without_improvement = 0
        checkpoint_path = artifacts_dir / f"fold_{fold_index}.pt"

        for epoch in range(1, args.epochs + 1):
            epoch_start = time.time()
            train_metrics = train_one_epoch(model, ema_model, train_loader, optimizer, scheduler, scaler, device, args)
            validation_metrics, _, _, _ = evaluate(model, validation_loader, device, tta=1)
            elapsed = time.time() - epoch_start

            print(
                f"fold={fold_index} epoch={epoch}/{args.epochs} "
                f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['accuracy']:.4f} "
                f"val_loss={validation_metrics['loss']:.4f} val_acc={validation_metrics['accuracy']:.4f} "
                f"time={elapsed:.1f}s"
            )

            if validation_metrics["accuracy"] > best_accuracy:
                best_accuracy = validation_metrics["accuracy"]
                best_epoch = epoch
                epochs_without_improvement = 0
                torch.save(
                    {
                        "model_state": copy.deepcopy(model.state_dict()),
                        "ema_state": None if ema_model is None else copy.deepcopy(ema_model.module.state_dict()),
                        "accuracy": best_accuracy,
                        "epoch": best_epoch,
                    },
                    checkpoint_path,
                )
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= args.patience:
                    print(f"fold={fold_index} early stopping at epoch {epoch}")
                    break

        checkpoint = torch.load(checkpoint_path, map_location=device)
        raw_model = SpokenDigitModel(args).to(device)
        raw_model.load_state_dict(checkpoint["model_state"])
        raw_metrics, _, raw_validation_probs, validation_labels = evaluate(raw_model, validation_loader, device, tta=args.tta)
        chosen_model = raw_model
        chosen_metrics = raw_metrics
        validation_probs = raw_validation_probs
        selected_model_name = "raw"

        if checkpoint.get("ema_state") is not None:
            ema_eval_model = SpokenDigitModel(args).to(device)
            ema_eval_model.load_state_dict(checkpoint["ema_state"])
            ema_metrics, _, ema_validation_probs, _ = evaluate(ema_eval_model, validation_loader, device, tta=args.tta)
            if (
                ema_metrics["accuracy"] > chosen_metrics["accuracy"]
                or (
                    ema_metrics["accuracy"] == chosen_metrics["accuracy"]
                    and ema_metrics["loss"] < chosen_metrics["loss"]
                )
            ):
                chosen_model = ema_eval_model
                chosen_metrics = ema_metrics
                validation_probs = ema_validation_probs
                selected_model_name = "ema"

        validation_metrics = chosen_metrics
        _, _, fold_test_probs, _ = evaluate(chosen_model, test_loader, device, tta=args.tta)
        oof_probabilities[validation_indices] = validation_probs
        oof_mask[validation_indices] = True
        test_probabilities += fold_test_probs / len(folds_to_run)

        fold_summary = {
            "fold": fold_index,
            "best_epoch": int(checkpoint["epoch"]),
            "best_val_accuracy": float(checkpoint["accuracy"]),
            "loaded_val_accuracy": float(validation_metrics["accuracy"]),
            "loaded_val_loss": float(validation_metrics["loss"]),
            "validation_size": int(len(validation_indices)),
            "test_size": int(len(test_ids)),
            "tta": int(args.tta),
            "ema_decay": float(args.ema_decay),
            "selected_model": selected_model_name,
            "raw_tta_val_accuracy": float(raw_metrics["accuracy"]),
        }
        fold_summaries.append(fold_summary)
        print(
            f"fold={fold_index} best_epoch={checkpoint['epoch']} "
            f"best_val_acc={checkpoint['accuracy']:.4f} reload_val_acc={validation_metrics['accuracy']:.4f}"
        )

        if validation_labels is not None:
            val_predictions = validation_probs.argmax(axis=1)
            per_fold_accuracy = float((val_predictions == validation_labels).mean())
            print(f"fold={fold_index} oof_accuracy={per_fold_accuracy:.4f}")

    oof_predictions = oof_probabilities.argmax(axis=1)
    covered_labels = np.asarray(labels)[oof_mask]
    covered_predictions = oof_predictions[oof_mask]
    oof_accuracy = float((covered_predictions == covered_labels).mean()) if covered_labels.size else 0.0
    submission = pd.DataFrame({"id": test_ids, "label": test_probabilities.argmax(axis=1).astype(int)})
    submission.to_csv(args.submission_path, index=False)

    oof_frame = train_df.copy()
    oof_frame["prediction"] = oof_predictions
    oof_frame["correct"] = (oof_frame["prediction"] == oof_frame["label"]).astype(int)
    oof_frame.to_csv(artifacts_dir / "oof_predictions.csv", index=False)
    np.save(artifacts_dir / "oof_probabilities.npy", oof_probabilities)
    np.save(artifacts_dir / "test_probabilities.npy", test_probabilities)

    summary = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "device": str(device),
        "preset": args.preset,
        "train_dir": str(train_dir),
        "test_dir": str(test_dir),
        "train_samples": int(len(train_df)),
        "test_samples": int(len(test_ids)),
        "folds_run": folds_to_run,
        "oof_coverage": int(oof_mask.sum()),
        "oof_accuracy": oof_accuracy,
        "folds": fold_summaries,
        "config": vars(args),
    }
    save_json(artifacts_dir / "cv_summary.json", summary)

    print(f"OOF accuracy: {oof_accuracy:.4f}")
    print(f"Wrote submission to {args.submission_path}")


#if __name__ == "__main__":
    #main()


In [2]:
from pathlib import Path
import sys

data_dir = Path("/kaggle/input")
output_dir = Path("/kaggle/working")

def find_competition_root(data_dir=Path("/kaggle/input")):
    for train_csv in data_dir.rglob("train.csv"):
        root = train_csv.parent
        if (root / "train_audio").exists() and (root / "test_audio").exists():
            return root
    raise FileNotFoundError(
        "Could not find a folder containing train.csv, train_audio/, and test_audio/ under /kaggle/input"
    )

competition_root = find_competition_root()
print("Using competition root:", competition_root)
print("Contents:", [p.name for p in competition_root.iterdir()])

sys.argv = [
    "train_spoken_digits.py",
    "--train-csv", str(competition_root / "train.csv"),
    "--train-dir", str(competition_root / "train_audio"),
    "--test-dir", str(competition_root / "test_audio"),
    "--submission-path", str(output_dir / "submission.csv"),
    "--artifacts-dir", str(output_dir / "artifacts"),
    "--device", "cuda",
]

sample_submission_path = competition_root / "sample_submission.csv"
if sample_submission_path.exists():
    sys.argv.extend(["--sample-submission", str(sample_submission_path)])

test_csv_path = competition_root / "test.csv"
if test_csv_path.exists():
    sys.argv.extend(["--test-csv", str(test_csv_path)])

main()


Using competition root: /kaggle/input/competitions/digitrecognition-ee708
Contents: ['sample_submission.csv', 'train_audio', 'test_audio', 'train.csv']
Training on 37800 samples across folds [0, 1, 2, 3, 4] using device=cuda
Resolved audio directories: train=/kaggle/input/competitions/digitrecognition-ee708/train_audio/train_audio test=/kaggle/input/competitions/digitrecognition-ee708/test_audio/test_audio
fold=0 epoch=1/18 train_loss=1.6556 train_acc=0.4603 val_loss=0.7508 val_acc=0.7714 time=182.5s
fold=0 epoch=2/18 train_loss=0.8366 train_acc=0.7774 val_loss=0.1520 val_acc=0.9724 time=48.0s
fold=0 epoch=3/18 train_loss=0.7057 train_acc=0.8096 val_loss=0.0961 val_acc=0.9823 time=42.6s
fold=0 epoch=4/18 train_loss=0.6233 train_acc=0.8165 val_loss=0.1236 val_acc=0.9824 time=37.1s
fold=0 epoch=5/18 train_loss=0.5945 train_acc=0.8412 val_loss=0.1537 val_acc=0.9811 time=38.4s
fold=0 epoch=6/18 train_loss=0.5993 train_acc=0.7957 val_loss=0.1247 val_acc=0.9878 time=41.5s
fold=0 epoch=7/18 t

In [3]:
import pandas as pd
pd.read_csv("/kaggle/working/submission.csv").head()

,id,label
0,test_000001,1
1,test_000002,6
2,test_000003,7
3,test_000004,1
4,test_000005,3
